In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 08 ? Multi-Hazard Triggers + Final Report
## Generates combined trigger matrices, risk summaries, and report-ready outputs


## Section 8.1 — FFWC Gauge Time Series

Time-series plots help place flood peaks and danger levels in hydrological context. This is the first step toward turning the mapping outputs into anticipatory-action logic.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

RAW_FFWC = RAW_DIR / 'ffwc'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
ffwc_df = pd.read_csv(RAW_FFWC / 'sylhet_stations_2024.csv')
ffwc_df['datetime'] = pd.to_datetime(ffwc_df.get('datetime', ffwc_df.get('date', pd.NaT)))
flood_area_df = pd.read_csv(REPORT_DIR / 'flood_area_by_date.csv')
flood_area_df['date'] = pd.to_datetime(flood_area_df['date'], format='%Y%m%d')
fig, ax = plt.subplots(figsize=(12, 5), dpi=300)
for station_name, station_df in ffwc_df.groupby('station'):
    value_col = 'water_level' if 'water_level' in station_df.columns else station_df.columns[-1]
    ax.plot(station_df['datetime'], station_df[value_col], label=station_name, linewidth=1.8)
    if 'danger_m' in station_df.columns:
        ax.axhline(station_df['danger_m'].iloc[0], linestyle='--', color='red', linewidth=0.8)
ax.axvline(pd.Timestamp('2024-06-19'), linestyle='--', color='black', linewidth=1, label='June peak')
ax.axvline(pd.Timestamp('2024-08-22'), linestyle=':', color='black', linewidth=1, label='August peak')
ax.set_title('FFWC gauge time series, May–September 2024')
ax.set_ylabel('Water level (m)')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'gauge_timeseries.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 8.2 — Gauge vs Flood Extent Alignment

Relating gauge levels to mapped flood area is the bridge between remote sensing and anticipatory action. The scatter plots here also provide the seventh evaluation diagram for the final submission package.


In [ ]:
regression_rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=300)
for ax, (station_name, station_df) in zip(axes.ravel(), ffwc_df.groupby('station')):
    value_col = 'water_level' if 'water_level' in station_df.columns else station_df.columns[-1]
    station_df = station_df.copy(); station_df['date'] = station_df['datetime'].dt.normalize()
    merged = flood_area_df.merge(station_df[['date', value_col]], on='date', how='left').dropna()
    if merged.empty:
        ax.set_visible(False)
        continue
    X = merged[[value_col]].values; y = merged['flooded_area_km2'].values
    reg = LinearRegression().fit(X, y); r2 = reg.score(X, y)
    regression_rows.append({'station': station_name, 'r2': r2})
    ax.scatter(merged[value_col], merged['flooded_area_km2'], color='#264653')
    xs = np.linspace(merged[value_col].min(), merged[value_col].max(), 100)
    ax.plot(xs, reg.predict(xs.reshape(-1, 1)), color='#e76f51')
    ax.set_title(f"{station_name} (R²={r2:.3f})")
    ax.set_xlabel('Gauge level (m)'); ax.set_ylabel('Flooded area (km²)')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'gauge_vs_flood_scatter.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)
regression_df = pd.DataFrame(regression_rows)
display(regression_df)


## Section 8.3 — Threshold Calibration

Turning inundation percentages into water-level thresholds creates the operational trigger table for watch, warning, and evacuation decisions. The thresholds here are transparent and easy for partners to refine.


In [ ]:
SYLHET_DIVISION_AREA_KM2 = 12635.0
trigger_specs = {'watch': 0.10 * SYLHET_DIVISION_AREA_KM2, 'warning': 0.25 * SYLHET_DIVISION_AREA_KM2, 'evacuation': 0.50 * SYLHET_DIVISION_AREA_KM2}
rows = []
for station_name, station_df in ffwc_df.groupby('station'):
    value_col = 'water_level' if 'water_level' in station_df.columns else station_df.columns[-1]
    station_df = station_df.copy(); station_df['date'] = station_df['datetime'].dt.normalize()
    merged = flood_area_df.merge(station_df[['date', value_col, 'danger_m']], on='date', how='left').dropna()
    if merged.empty:
        continue
    peak_date = merged.loc[merged['flooded_area_km2'].idxmax(), 'date']
    result = {'station': station_name, 'danger_m': float(merged['danger_m'].dropna().iloc[0]) if 'danger_m' in merged and merged['danger_m'].notna().any() else np.nan}
    for trigger_name, flood_thresh in trigger_specs.items():
        eligible = merged[merged['flooded_area_km2'] >= flood_thresh]
        result[f'{trigger_name}_m'] = float(eligible[value_col].min()) if not eligible.empty else np.nan
        result[f'lead_{trigger_name}_hrs'] = float((peak_date - eligible.iloc[0]['date']).total_seconds() / 3600) if not eligible.empty else np.nan
    result['lead_A_hrs'] = result.pop('lead_watch_hrs')
    result['lead_B_hrs'] = result.pop('lead_warning_hrs')
    result['lead_C_hrs'] = result.pop('lead_evacuation_hrs')
    result['recommended_action'] = 'Escalate alerts and pre-position child-centred services'
    rows.append(result)
thresholds_df = pd.DataFrame(rows)
thresholds_df.to_csv(REPORT_DIR / 'action_thresholds.csv', index=False)
display(thresholds_df)


## Section 8.4 — Threshold Visualization

The threshold plot shows how the selected gauge values separate normal, watch, warning, and evacuation conditions. It is one of the clearest figures for explaining the anticipatory-action logic to non-technical reviewers.


In [ ]:
if not thresholds_df.empty:
    row = thresholds_df.iloc[0]
    station_df = ffwc_df[ffwc_df['station'] == row['station']].copy()
    value_col = 'water_level' if 'water_level' in station_df.columns else station_df.columns[-1]
    station_df['date'] = station_df['datetime'].dt.normalize()
    merged = flood_area_df.merge(station_df[['date', value_col]], on='date', how='left').dropna()
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    ax.plot(merged['date'], merged[value_col], color='#1d3557', linewidth=2)
    for key, color in [('watch_m', 'gold'), ('warning_m', 'orange'), ('evacuation_m', 'red')]:
        if pd.notna(row[key]):
            ax.axhline(row[key], linestyle='--', color=color, linewidth=1.2, label=key.replace('_m', '').upper())
    ax.set_title(f"Threshold zones for {row['station']}"); ax.set_ylabel('Gauge level (m)'); ax.legend()
    plt.tight_layout(); plt.savefig(FIGURES_DIR / 'action_thresholds.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 8.5 — Child-Centred Trigger Matrix

The trigger matrix turns thresholds into child-facing actions such as school pre-closure and clinic evacuation. This is the most competition-specific output in the entire pipeline.


In [ ]:
impact_df = pd.read_csv(REPORT_DIR / 'infrastructure_impact.csv')
top_unions = impact_df.sort_values('children_exposed', ascending=False).head(10)
ref = thresholds_df.iloc[0] if not thresholds_df.empty else None
trigger_matrix = pd.DataFrame([
    {'Trigger Level': 'WATCH', 'Gauge m': float(ref['watch_m']) if ref is not None and pd.notna(ref['watch_m']) else np.nan, 'Schools Pre-Close': ', '.join(top_unions['union_name'].head(10).astype(str)), 'Shelters Activate': ', '.join(top_unions['union_name'].head(5).astype(str)), 'Clinics Evacuate': ', '.join(top_unions['union_name'].head(5).astype(str))},
    {'Trigger Level': 'WARNING', 'Gauge m': float(ref['warning_m']) if ref is not None and pd.notna(ref['warning_m']) else np.nan, 'Schools Pre-Close': 'All affected unions', 'Shelters Activate': 'All affected unions', 'Clinics Evacuate': 'All affected unions'},
    {'Trigger Level': 'EVACUATION', 'Gauge m': float(ref['evacuation_m']) if ref is not None and pd.notna(ref['evacuation_m']) else np.nan, 'Schools Pre-Close': 'All Sylhet', 'Shelters Activate': 'All Sylhet', 'Clinics Evacuate': 'All Sylhet'},
])
trigger_matrix.to_csv(REPORT_DIR / 'child_trigger_matrix.csv', index=False)
display(trigger_matrix)


## Section 8.6 — Final Report Generation

The final markdown and CSV outputs gather the whole pipeline into a submission-ready package: architecture, ablation, performance, flood progression, impact, thresholds, and recommendations.


In [ ]:
ablation_df = pd.read_csv(REPORT_DIR / 'ablation_results.csv')
scene_df = pd.read_csv(REPORT_DIR / 'scene_accuracy_assessment.csv') if (REPORT_DIR / 'scene_accuracy_assessment.csv').exists() else pd.DataFrame()
impact_df = pd.read_csv(REPORT_DIR / 'infrastructure_impact.csv')
top5 = impact_df.sort_values('children_exposed', ascending=False).head(5)
report_md = f"""# Sylhet Flood 2024 CASA-Net Impact Summary

## 1. Executive Summary
CASA-Net was configured as an end-to-end, uncertainty-aware SAR flood mapping pipeline for the 2024 Sylhet floods.

## 2. CASA-Net Architecture Summary
CASA-Net combines asymmetric VV/VH encoding, CPAG fusion, terrain-conditioned decoding with FiLM, and MC Dropout uncertainty estimation.

## 3. Ablation Study Results Table
{ablation_df.to_markdown(index=False)}

## 4. Model Performance Table
{scene_df.to_markdown(index=False) if not scene_df.empty else 'Scene assessment pending.'}

## 5. Flood Progression Table
{flood_area_df.to_markdown(index=False)}

## 6. Infrastructure Impact Table
{impact_df.head(20).to_markdown(index=False)}

## 7. Uncertainty-Flagged Facilities Table
{impact_df[['union_name', 'uncertain_facilities']].head(20).to_markdown(index=False)}

## 8. Anticipatory Action Threshold Table
{thresholds_df.to_markdown(index=False) if not thresholds_df.empty else 'Threshold calibration pending.'}

## 9. Child-Centred Trigger Matrix
{trigger_matrix.to_markdown(index=False)}

## 10. Recommendations
Top 5 unions for pre-positioning by children exposed: {', '.join(top5['union_name'].astype(str).tolist())}
"""
(REPORT_DIR / 'flood_impact_summary.md').write_text(report_md, encoding='utf-8')
impact_df.to_csv(REPORT_DIR / 'flood_impact_summary.csv', index=False)
print(REPORT_DIR / 'flood_impact_summary.md')


## Section 8.7 — Competition Submission Checklist

This checklist is the final handoff sanity check before packaging the project for submission.


In [ ]:
checklist = [
    'Track 01 ResilienceAI: infrastructure exposure + automated pipeline',
    'Novel architecture: CASA-Net (asymmetric encoder + CPAG + HTC + uncertainty)',
    'Ablation study: 4 variants proving each component's contribution',
    'Child-centred framing: children_exposed + Child Protection Hub validation',
    'Anticipatory action trigger matrix: gauge thresholds -> school pre-closure list',
    'Uncertainty-aware impact assessment',
    'Reproducibility: SEED=42, split_index.json, all intermediates saved',
    'GitHub-ready: 8 ordered notebooks, runs top-to-bottom independently',
    'Figures: all at 300 DPI, ready for 10-slide deck',
]
for item in checklist:
    print(f'✅ {item}')


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 8.8 ? Multi-Hazard Trigger Matrix and Report Extension
This section upgrades the flood-only trigger framework into a **multi-hazard anticipatory action matrix**. Flood, erosion, and landslide probabilities can be merged into a joint action table that prioritises unions and facilities under compounding risk.


In [ ]:
    # MULTI-HAZARD EXTENSION GENERATED
    import pandas as pd
    from analysis.multi_hazard_support import load_hazard_catalog

    ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
    hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')

    trigger_rows = []
    for hazard in hazard_catalog['hazards']:
        trigger_rows.extend([
            {'hazard': hazard['id'], 'trigger_level': 'WATCH', 'trigger_rule': 'combined_risk >= 0.20', 'recommended_action': f"Prepare {hazard['name'].lower()} monitoring and pre-positioning"},
            {'hazard': hazard['id'], 'trigger_level': 'WARNING', 'trigger_rule': 'combined_risk >= 0.40', 'recommended_action': f"Activate targeted {hazard['name'].lower()} response in top-risk unions"},
            {'hazard': hazard['id'], 'trigger_level': 'EMERGENCY', 'trigger_rule': 'combined_risk >= 0.60', 'recommended_action': f"Escalate {hazard['name'].lower()} actions and protect critical child-centred services"},
        ])
    multi_hazard_trigger_matrix = pd.DataFrame(trigger_rows)
    multi_hazard_trigger_matrix.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_trigger_matrix.csv', index=False)

    summary_md = [
        '# Multi-Hazard Summary',
        '',
        'This project now supports a shared flood, erosion, and landslide workflow.',
        '',
        multi_hazard_trigger_matrix.to_markdown(index=False),
    ]
    (ROOT / 'outputs' / 'report' / 'multi_hazard_summary.md').write_text('
'.join(summary_md), encoding='utf-8')
    multi_hazard_trigger_matrix
